In [ ]:
!pip install pdfplumber openpyxl pandas

# 自動化填寫交易資料至流水帳

In [ ]:
import re
import pandas as pd
import numpy as np
import pdfplumber
from io import BytesIO
from openpyxl import load_workbook
from google.colab import files
from openpyxl.styles import Font

import re
import pandas as pd

# 正則式定義：用於解析金額、日期與時間格式
MONEY_RE = re.compile(r"\d{1,3}(?:,\d{3})*(?:\.\d{2})?")
DATE_RE = re.compile(r"\d{4}/\d{2}/\d{2}")
TIME_RE = re.compile(r"\d{2}:\d{2}:\d{2}")

# 從 token 中還原金額：將分段的金額字串合併並轉為 float
def recover_money_tokens(tokens):
    try:
        money_parts = [t for t in tokens if re.match(r'^[\d,.]+$', t)]
        joined = ''.join(money_parts)
        found = MONEY_RE.findall(joined)
        return [float(m.replace(',', '')) for m in found]
    except Exception:
        return []

# 格式化金額：將 float 轉為千分位整數字串
def format_amount(value):
    try:
        return "{:,}".format(int(float(value)))
    except (ValueError, TypeError):
        return value

# 套用金額格式化至指定欄位
def format_money_fields(df, columns=['支出', '收入', '餘額']):
    for col in columns:
        if col in df.columns:
            df[col] = df[col].apply(format_amount)
    return df

# 玉山銀行 187 分行解析器：解析每行交易資料
def parse_yushan_187(lines):
    recs = []
    for line in lines:
        line = str(line).strip()
        if not line or '總計' in line or 'page:' in line or '提' in line:
            continue

        tokens = line.split()
        if len(tokens) < 9:
            continue  # 欄位太少，不處理

        try:
            date = pd.to_datetime(tokens[1], errors='coerce')
            summary = tokens[4]
            支出 = float(tokens[5].replace(',', '')) if tokens[5] != '0' else 0.0
            收入 = float(tokens[6].replace(',', '')) if tokens[6] != '0' else 0.0
            餘額 = float(tokens[7].replace(',', ''))
            raw_client = ' '.join(tokens[8:]).strip()

            # 插入分隔線：將第一個空格分成前後段
            if ' ' in raw_client:
                parts = raw_client.split(' ', 1)
                客戶 = f"{parts[0]}｜{parts[1]}"
            else:
                客戶 = raw_client  # 若無空格則不處理

            recs.append({
                '帳務日期': date,
                '明細': summary,
                '支出': 支出,
                '收入': 收入,
                '餘額': 餘額,
                '客戶/供應商': 客戶
            })
        except Exception:
            continue

    df = pd.DataFrame(recs)
    df = format_money_fields(df)
    print(f"\n✅ [玉山] 成功解析筆數：{len(df)}")
    return df

# 兆豐銀行 703 分行解析器：解析每行交易資料
def parse_mega_703(lines):
    DATE_RE = re.compile(r'\d{4}/\d{2}/\d{2}')
    MONEY_RE = re.compile(r'\d{1,3}(?:,\d{3})*\.\d{2}')

    def is_noise_line(line):
        return bool(re.search(
            r'本頁合計|累計|Mega International|存款明細表|查詢人員|查詢時間|帳號|設帳行|戶名|第\d頁/共\d頁',
            line
        ))

    def clean_detail_703(text):
        text = re.sub(r'^\d{4}/\d{2}/\d{2}', '', text)
        text = re.sub(r'\d{1,3}(?:,\d{3})*\.\d{2}', '', text)
        text = re.sub(r'\d{4}/\d{2}/\d{2}\(\d{2}:\d{2}:\d{2}\)', '', text)
        text = re.sub(r'本頁合計|累計|Mega International.*|查詢人員.*|查詢時間.*|帳號.*|戶名.*|設帳行.*|第\d頁/共\d頁', '', text)
        return text.strip()

    def finalize_entry(entry, main_detail_line):
        entry['明細'] = clean_detail_703(main_detail_line)
        return entry

    recs = []
    current_entry = None
    main_detail_line = ''

    def is_main_trade_703(tokens):
        return any(DATE_RE.match(t) for t in tokens) and any(MONEY_RE.match(t) for t in tokens)

    for line in lines:
        line = str(line).strip()
        if not line or is_noise_line(line) or re.search(r'帳務日期|摘要|存入金額|支出金額|帳戶餘額|交易日期', line):
            continue

        tokens = line.split()
        money_tokens = [t for t in tokens if MONEY_RE.match(t)]

        if is_main_trade_703(tokens):
            if current_entry:
                recs.append(finalize_entry(current_entry, main_detail_line))

            date = next((pd.to_datetime(t, errors='coerce') for t in tokens if DATE_RE.match(t)), pd.NaT)
            amount = float(money_tokens[0].replace(',', '')) if money_tokens else 0.0
            收入, 支出 = 0.0, amount
            餘額 = ''
            if len(money_tokens) == 2 and amount == 0.0:
                收入 = float(money_tokens[1].replace(',', ''))
                支出 = 0.0
            elif len(money_tokens) == 2:
                餘額 = float(money_tokens[1].replace(',', ''))

            current_entry = {
                '帳務日期': date,
                '客戶/供應商': '',
                '明細': '',
                '支出': 支出,
                '收入': 收入,
                '餘額': 餘額
            }
            main_detail_line = line  # 只保留主行
        else:
            continue  # 忽略延伸行

    if current_entry:
        recs.append(finalize_entry(current_entry, main_detail_line))

    df = pd.DataFrame(recs)
    df = format_money_fields(df)
    print(f"\n✅ [兆豐] 成功解析筆數：{len(df)}")
    return df

# 兆豐銀行 347 分行解析器：解析每行交易資料
def parse_mega_347(lines):
    MONEY_RE = re.compile(r'\d{1,3}(?:,\d{3})*\.\d{2}')
    DATE_RE = re.compile(r'^\d{4}/\d{2}/\d{2}')
    DATETIME_RE = re.compile(r'\d{4}/\d{2}/\d{2}\(\d{2}:\d{2}:\d{2}\)')

    def clean_detail_347(text):
        text = re.sub(DATE_RE, '', text)                        # 去開頭日期
        text = re.sub(MONEY_RE, '', text)                       # 去金額
        text = re.sub(DATETIME_RE, '', text)                    # 去時間戳
        text = re.sub(DATETIME_RE, '', text)
        return text.strip()

    recs = []
    buffer = []
    current_entry = None

    def is_main_trade_347(tokens):
        return any(DATE_RE.match(t) for t in tokens) and any(MONEY_RE.match(t) for t in tokens)

    for line in lines:
        line = str(line).strip()
        if not line or re.search(r'帳務日期|摘要|存入金額|支出金額|帳戶餘額|交易日期', line):
            continue

        tokens = line.split()
        money_tokens = [t for t in tokens if MONEY_RE.match(t)]

        if is_main_trade_347(tokens):
            if current_entry:
                current_entry['明細'] = '｜'.join(buffer)
                recs.append(current_entry)

            date = next((pd.to_datetime(t, errors='coerce') for t in tokens if DATE_RE.match(t)), pd.NaT)
            amount = float(money_tokens[0].replace(',', ''))
            收入, 支出 = 0.0, amount
            餘額 = ''
            if len(money_tokens) == 2:
                餘額 = float(money_tokens[1].replace(',', ''))
            elif amount == 0.0 and len(money_tokens) >= 2:
                收入 = float(money_tokens[1].replace(',', ''))
                支出 = 0.0

            current_entry = {
                '帳務日期': date if not pd.isna(date) else '',
                '客戶/供應商': '',
                '明細': '',
                '支出': 支出,
                '收入': 收入,
                '餘額': 餘額
            }
            buffer = [clean_detail_347(line)]
        else:
            buffer.append(clean_detail_347(line))

    if current_entry:
        current_entry['明細'] = '｜'.join(buffer)
        recs.append(current_entry)

    df = pd.DataFrame(recs)
    df = format_money_fields(df)
    print(f"\n✅ [兆豐] 成功解析筆數：{len(df)}")
    return df

# 兆豐銀行 182 分行解析器：解析每行交易資料
def parse_mega_182(lines):
    MONEY_RE = re.compile(r'\d{1,3}(?:,\d{3})*\.\d{2}')
    DATE_RE = re.compile(r'^\d{4}/\d{2}/\d{2}')
    DATETIME_RE = re.compile(r'\d{4}/\d{2}/\d{2}\(\d{2}:\d{2}:\d{2}\)')

    def clean_detail_182(text):
        text = re.sub(DATE_RE, '', text)                        # 去開頭日期
        text = re.sub(MONEY_RE, '', text)                       # 去金額
        text = re.sub(DATETIME_RE, '', text)                    # 去時間戳
        return text.strip()

    recs = []
    buffer = []
    current_entry = None

    def is_main_trade_182(tokens):
        return any(DATE_RE.match(t) for t in tokens) and any(MONEY_RE.match(t) for t in tokens)

    for line in lines:
        line = str(line).strip()
        if not line or re.search(r'帳務日期|摘要|存入金額|支出金額|帳戶餘額|交易日期', line):
            continue

        tokens = line.split()
        money_tokens = [t for t in tokens if MONEY_RE.match(t)]

        if is_main_trade_182(tokens):
            if current_entry:
                current_entry['明細'] = '｜'.join(buffer)
                recs.append(current_entry)

            date = next((pd.to_datetime(t, errors='coerce') for t in tokens if DATE_RE.match(t)), pd.NaT)
            amount = float(money_tokens[0].replace(',', ''))
            收入, 支出 = 0.0, amount
            餘額 = ''
            if len(money_tokens) == 2:
                餘額 = float(money_tokens[1].replace(',', ''))
            elif amount == 0.0 and len(money_tokens) >= 2:
                收入 = float(money_tokens[1].replace(',', ''))
                支出 = 0.0

            current_entry = {
                '帳務日期': date if not pd.isna(date) else '',
                '客戶/供應商': '',
                '明細': '',
                '支出': 支出,
                '收入': 收入,
                '餘額': 餘額
            }
            buffer = [clean_detail_182(line)]
        else:
            buffer.append(clean_detail_182(line))

    if current_entry:
        current_entry['明細'] = '｜'.join(buffer)
        recs.append(current_entry)

    df = pd.DataFrame(recs)
    df = format_money_fields(df)
    print(f"\n✅ [兆豐] 成功解析筆數：{len(df)}")
    return df

# 兆豐銀行 697 分行解析器：解析每行交易資料
def parse_mega_697(lines):
    MONEY_RE = re.compile(r'\d{1,3}(?:,\d{3})*\.\d{2}')
    DATE_RE = re.compile(r'^\d{4}/\d{2}/\d{2}')
    DATETIME_RE = re.compile(r'\d{4}/\d{2}/\d{2}\(\d{2}:\d{2}:\d{2}\)')

    def clean_detail_697(text):
        text = re.sub(DATE_RE, '', text)                        # 去開頭日期
        text = re.sub(MONEY_RE, '', text)                       # 去金額
        text = re.sub(DATETIME_RE, '', text)                    # 去時間戳
        return text.strip()

    recs = []
    buffer = []
    current_entry = None

    def is_main_trade_697(tokens):
        return any(DATE_RE.match(t) for t in tokens) and any(MONEY_RE.match(t) for t in tokens)

    for line in lines:
        line = str(line).strip()
        if not line or re.search(r'帳務日期|摘要|存入金額|支出金額|帳戶餘額|交易日期', line):
            continue
        # 排除頁尾、報表資訊、查詢人員等非交易行
        if re.search(r'本頁合計|累計|Mega International|存款明細表|查詢人員|查詢時間|帳號|設帳行|戶名|第\d頁/共\d頁', line):
            continue

        tokens = line.split()
        money_tokens = [t for t in tokens if MONEY_RE.match(t)]

        if is_main_trade_697(tokens):
            if current_entry:
                current_entry['明細'] = '｜'.join(buffer)
                recs.append(current_entry)

            date = next((pd.to_datetime(t, errors='coerce') for t in tokens if DATE_RE.match(t)), pd.NaT)
            amount = float(money_tokens[0].replace(',', ''))
            收入, 支出 = 0.0, amount
            餘額 = ''
            if len(money_tokens) == 2:
                餘額 = float(money_tokens[1].replace(',', ''))
            elif amount == 0.0 and len(money_tokens) >= 2:
                收入 = float(money_tokens[1].replace(',', ''))
                支出 = 0.0

            current_entry = {
                '帳務日期': date if not pd.isna(date) else '',
                '客戶/供應商': '',
                '明細': '',
                '支出': 支出,
                '收入': 收入,
                '餘額': 餘額
            }
            buffer = [clean_detail_697(line)]
        else:
            buffer.append(clean_detail_697(line))

    if current_entry:
        current_entry['明細'] = '｜'.join(buffer)
        recs.append(current_entry)

    df = pd.DataFrame(recs)
    df = format_money_fields(df)
    print(f"\n✅ [兆豐] 成功解析筆數：{len(df)}")
    return df

# 主匯入函式：從 PDF 解析帳務資料並寫入 Excel
def 匯入(目標月份=4):
    upl_exc = files.upload()
    upl_pdf = files.upload()
    wb = load_workbook(list(upl_exc.keys())[0])
    total = 0

    for fn, fb in upl_pdf.items():
        bank = fn.split('_')[0]
        print(f"\n📄 處理檔案：{fn}（帳戶別：{bank}）")

        # 解析 PDF 取得所有行
        lines = []
        with pdfplumber.open(BytesIO(fb)) as pdf:
            for i, p in enumerate(pdf.pages):
                text = p.extract_text()
                print(f"\n📄 Page {i+1}:\n{text}")  # 看看有沒有抓到文字
                lines += (text or '').split('\n')

        # 1. 定義一張對照表
        parse_map = {
            '玉山187': parse_yushan_187,
            '兆豐347': parse_mega_347,
            '兆豐703': parse_mega_703,
            '兆豐182': parse_mega_182,
            '兆豐697': parse_mega_697,
            # 若未來有更多，只要再加一行
        }

        # 2. 取得 parser
        parser = parse_map.get(bank)
        if not parser:
            print(f"⚠️ 未支援的銀行別：{bank}")
            df = pd.DataFrame()
        else:
            df = parser(lines)

        # 日期欄轉型與失敗筆數偵測
        df['帳務日期'] = pd.to_datetime(df['帳務日期'], errors='coerce')
        df_failed = df[df['帳務日期'].isna()]
        if not df_failed.empty:
            print(f"⚠️ 日期轉換失敗筆數：{len(df_failed)}")
            print(df_failed[['帳務日期', '明細']].to_string(index=False))

        # 篩選指定月份
        df3 = df[df['帳務日期'].dt.month == 目標月份].copy()
        df3['餘額'] = df3['餘額'].bfill()
        print(f"[{bank}] 篩 {目標月份} 月共 {len(df3)} 筆")
        print(df3.head(5).to_string(index=False))

        # 清空原始 Excel 資料
        ws = wb[bank]
        for r in range(5, ws.max_row + 1):
            for c in range(1, 7):
                ws.cell(row=r, column=c, value='')

        row = 3
        for _, rec in df3.iterrows():
            ws.cell(row=row, column=1, value=rec['帳務日期'].strftime('%Y-%m-%d') if pd.notna(rec['帳務日期']) else '').font = Font(color="000000")
            ws.cell(row=row, column=2, value=rec.get('客戶/供應商', '')).font = Font(color="000000")
            ws.cell(row=row, column=3, value=rec.get('明細', '')).font = Font(color="000000")
            ws.cell(row=row, column=4, value=format_amount(rec["支出"])).font = Font(color="000000")
            ws.cell(row=row, column=5, value=format_amount(rec["收入"])).font = Font(color="000000")
            ws.cell(row=row, column=6, value=format_amount(rec["餘額"])).font = Font(color="000000")
            row += 1
            total += 1

    out = 'out.xlsx'
    wb.save(out)
    print(f"\n✅ Excel匯入完成，共寫入 {total} 筆資料")
    return wb  # ✅ 回傳工作簿物件供後續使用
wb = 匯入(4)

Saving 巨量流水帳_4月.xlsx to 巨量流水帳_4月 (3).xlsx


Saving 玉山187_月對帳單.pdf to 玉山187_月對帳單 (1).pdf
Saving 兆豐182_月對帳單.pdf to 兆豐182_月對帳單 (1).pdf
Saving 兆豐347_月對帳單.pdf to 兆豐347_月對帳單 (1).pdf
Saving 兆豐697_月對帳單.pdf to 兆豐697_月對帳單 (1).pdf
Saving 兆豐703_月對帳單.pdf to 兆豐703_月對帳單 (1).pdf

📄 處理檔案：玉山187_月對帳單 (1).pdf（帳戶別：玉山187）

📄 Page 1:
交易名稱 交易明細查詢
查詢人員 授權管理人員 查詢時間 2025/07/02 11:07:20(GMT+8)
顧客ID 24616337 巨量移動科技有限公司 幣別 臺幣
帳號 0794940086187 活期存款 查詢依據 交易日
查詢區間 2025/01/01 00:00:00 ~ 2025/07/02 24:00:00 資料排序 舊到新
實際交易時
序號 帳務日期 實際交易日期 摘要 提 存 餘額 備註 轉出入銀行代號/帳號
間
1 2025/01/06 2025/01/06 01:49:59 ＦＸＭＬ入帳 0 1,050,000 2,865,992 財團法人資訊工業策進會 017/0000003009016889
2 2025/01/07 2025/01/07 18:53:46 中華電信費用 5,500 0 2,860,492 臺銀 004/0099995100400543
3 2025/01/08 2025/01/08 12:20:46 中華電信費用 4,125 0 2,856,367 臺銀 004/0099995100400543
4 2025/01/08 2025/01/08 12:22:52 中華電信費用 999 0 2,855,368 臺銀 004/0099995100400543
5 2025/01/20 2025/01/20 21:59:23 全民健康保險 13,228 0 2,842,140 臺銀 004/0099991459520543
6 2025/02/03 2025/02/03 18:36:04 全民健康保險 453 0 2,841,687 臺銀 004/0099991459520543
7 2025/0

# 轉帳明細更新版

In [ ]:
import pandas as pd
from datetime import datetime
from openpyxl import load_workbook
from IPython.display import display
from google.colab import files

# 將值安全轉為字串，避免 NaN 造成錯誤
def safe_str(val):
    return "" if pd.isna(val) else str(val).strip()

# 清理金額欄位：移除逗號與非數字符號，轉為 float
def clean_amount_column(series):
    return (
        series.astype(str)
        .str.replace(",", "", regex=False)       # 移除千分位逗號
        .str.replace(" ", "", regex=False)       # 移除空白
        .str.replace("(", "-", regex=False)      # 括號負號轉換
        .str.replace(")", "", regex=False)
        .replace("", None)                       # 空字串轉 NaN
        .astype(float)                           # 最終轉 float
    )

# 比對交易資料：根據日期與金額誤差判斷是否為相符交易
def match_transaction(row, tran, amount_tolerance=30):
    # 日期欄位防呆
    if pd.isna(row.get("帳務日期")) or pd.isna(tran.日期):
        return False

    # 日期格式化比對
    try:
        tran_date = tran.日期.strftime("%Y-%m-%d")
    except Exception as e:
        print(f"❌ 日期格式錯誤：{e}")
        return False

    # 金額比對
    for col in ["支出", "收入"]:
        if pd.notna(row.get(col)):
            try:
                diff = abs(row[col] - tran.交易金額)
                print(f"🔍 金額比對：row={row[col]} vs tran={tran.交易金額}｜差額={diff}")
                if diff <= amount_tolerance:
                    return True
            except Exception as e:
                print(f"❌ 金額格式錯誤：{e}")
                return False

    # 若支出/收入都無法比對成功
    return False

def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

# 更新交易資料列：補上受款人與備註資訊
def update_row_content(df, idx, tran):
    updated = False
    raw = tran.to_dict()

    # 無條件寫入受款人到「客戶/供應商」
    payee = safe_str(raw.get("受款人", ""))
    df.at[idx, "客戶/供應商"] = payee
    updated = True

    # 補上備註（避免重複）
    remark = safe_str(raw.get("備註", ""))
    if remark:
        original_detail = safe_str(df.at[idx, "明細"])
        if original_detail and remark not in original_detail:
            df.at[idx, "明細"] = f"{original_detail}｜{remark}"
            updated = True
        elif not original_detail:
            df.at[idx, "明細"] = remark
            updated = True

    return updated

def safe_update_sheet(ws, sheet_name, df_transfer_subset, expected_columns, amount_tolerance=30):
    # 1. 讀取分頁 & 欄位清理
    df_sheet = pd.read_excel("out.xlsx", sheet_name=sheet_name, skiprows=1)
    print(df_sheet.columns.tolist())
    df_sheet.columns = (
        df_sheet.columns.astype(str)
        .str.strip()
        .str.replace("　", "", regex=False)
    )



    # 2. 轉型帳務日期 → datetime → 向下補值
    df_sheet["帳務日期"] = (
        pd.to_datetime(df_sheet["帳務日期"], errors="coerce")
        .ffill()
    )

    # 3. 清洗支出／收入欄位為純 float
    for col in ["支出", "收入"]:
        if col in df_sheet.columns:
            df_sheet[col] = clean_amount_column(df_sheet[col])

    # 4. 篩選非空行
    df_sheet = df_sheet[
        df_sheet["帳務日期"].notna() |
        df_sheet["支出"].notna() |
        df_sheet["收入"].notna()
    ]
    missing_cols = [col for col in ["帳務日期", "客戶/供應商", "支出"] if col not in df_sheet.columns]
    if missing_cols:
        print(f"❗ 欄位缺失：{missing_cols}，請確認 Excel 標題列是否正確")


    # 5. 保留物件欄位但**不要**包含 帳務日期
    for col in ["客戶/供應商", "明細", "有無憑證(憑證編號)"]:
        if col in df_sheet.columns:
            df_sheet[col] = df_sheet[col].astype("object")

    # 6. 比對函式：改用 isinstance 檢查 scalar
    def match_transaction_by_tolerance(row, df_trans, amount_tolerance=30, date_tolerance=1):
        dt = row["帳務日期"]
        amt = row["支出"] if pd.notna(row["支出"]) else row["收入"]

        if not isinstance(dt, pd.Timestamp) or pd.isna(amt):
            return None

        # 🔧 統一金額精度
        amt = round(amt)
        df_trans["交易金額"] = df_trans["交易金額"].round()

        # ✅ Step 1：先篩金額在容差範圍內
        candidates = df_trans[
            (df_trans["交易金額"] - amt).abs() <= amount_tolerance
        ]

        if candidates.empty:
            print(f"❌ 無候選交易：金額={amt}（±{amount_tolerance}）")
            return None

        # ✅ Step 2：再篩日期在 ±1 天內
        candidates = candidates[
            (candidates["日期"] >= dt - pd.Timedelta(days=date_tolerance)) &
            (candidates["日期"] <= dt + pd.Timedelta(days=date_tolerance))
        ]

        if candidates.empty:
            print(f"❌ 無候選交易：日期={dt.date()}（±{date_tolerance} 天） 金額={amt}")
            return None

        # ✅ 回傳第一筆符合者（不比對受款人）
        return candidates.iloc[0]

    # 7. 補寫迴圈
    match_count = 0
    for idx, row in df_sheet.iterrows():
        tran = match_transaction_by_tolerance(row, df_transfer_subset, amount_tolerance)
        if tran is not None and not tran.empty:
            update_row_content(df_sheet, idx, tran)
            match_count += 1
        else:
            print(f"🔍 第 {idx} 筆無法比對，略過")
    # 8. 一次性格式化帳務日期為 YYYY-MM-DD
    df_sheet["帳務日期"] = df_sheet["帳務日期"].dt.strftime("%Y-%m-%d")

    # 9. 前向填補其他欄位
    fill_cols = ["帳務日期", "客戶/供應商", "明細", "有無憑證(憑證編號)"]
    existing = [c for c in fill_cols if c in df_sheet.columns]
    df_sheet[existing] = df_sheet[existing].ffill().infer_objects(copy=False)

    # 10. 最後一次性寫回 Excel
    for r_idx, row in enumerate(df_sheet.itertuples(index=False), start=3):
        for c_idx, val in enumerate(row[:len(expected_columns)], start=1):
            ws.cell(row=r_idx, column=c_idx, value=val if val != "" else None)
    # 11. 套用三位一撇格式（千分位格式）
    for row in ws.iter_rows(min_row=3, min_col=4, max_col=6):  # 假設支出=第4欄，收入=第5欄，餘額=第6欄
        for cell in row:
            try:
                float(cell.value)  # 確保是數字
                cell.number_format = '#,##0'  # 或 '#,##0.00' 若你要保留小數
            except:
                continue

# 主流程：上傳兆豐轉帳明細並依帳戶分頁補寫交易資料
def update_megabank_transfer_details(wb, target_month=4, output_path='完整金流整合.xlsx'):
    import openpyxl
    from openpyxl.styles import numbers

    print("📁 請上傳兆豐轉帳明細 Excel")
    uploaded = files.upload()
    transfer_excel = list(uploaded.keys())[0]
    df_transfer = pd.read_excel(transfer_excel)

    # 1. 確保日期欄位存在且轉型
    if "日期" not in df_transfer.columns:
        print("❌ 未找到『日期』欄位，無法篩選月份")
        return
    df_transfer["日期"] = pd.to_datetime(df_transfer["日期"], errors="coerce")

    # 2. 篩選指定月份
    df_transfer = df_transfer[df_transfer["日期"].dt.month == target_month]

    # 3. 檢核必要欄位
    required_cols = ["出帳帳戶", "交易金額", "日期"]
    missing = [c for c in required_cols if c not in df_transfer.columns]
    if missing:
        print(f"⚠️ 缺少必要欄位：{missing}")
        return

    # 4. 前置顯示與清洗
    display(df_transfer.head())
    df_transfer["交易金額"] = pd.to_numeric(df_transfer["交易金額"], errors="coerce")

    # 5. 提取帳戶末三碼
    def extract_suffix(account):
        try:
            s = str(int(float(account)))
            return s[-3:].zfill(3)
        except:
            return None

    df_transfer["出帳帳戶末三碼"] = df_transfer["出帳帳戶"].apply(extract_suffix)
    print("📊 末三碼分布：")
    print(df_transfer["出帳帳戶末三碼"].value_counts())

    # 6. 準備各分頁對應
    valid_sheets = {}
    for name in wb.sheetnames:
        if name.startswith("兆豐"):
            suffix = name[-3:]
            valid_sheets[suffix] = name

    expected_cols = [
        "帳務日期", "客戶/供應商", "明細",
        "支出", "收入", "餘額",
        "有無憑證(憑證編號)"
    ]

    # 7. 執行補寫
    for suffix, sheet_name in valid_sheets.items():
        subset = df_transfer[df_transfer["出帳帳戶末三碼"] == suffix]
        if subset.empty:
            print(f"🔕 無資料：{sheet_name} ({suffix})，略過")
            continue
        ws = wb[sheet_name]
        safe_update_sheet(ws, sheet_name, subset, expected_cols)

    # 9. 儲存檔案
    wb.save(output_path)
    print(f"✅ 完成！格式化後的檔案已存為：{output_path}")
# 使用方式
wb = load_workbook("out.xlsx")
update_megabank_transfer_details(wb)
wb.save("完整金流整合.xlsx")

📁 請上傳兆豐轉帳明細 Excel


Saving 兆豐703網銀轉帳明細_參考.xlsx to 兆豐703網銀轉帳明細_參考 (50).xlsx


,日期,出帳帳戶,受款人銀行帳號,交易金額,受款人,備註,Unnamed: 6
13,2025-04-07,21009161703,20310608786,6670,RENALDI EGA HENDRAWAN梁民志 ...,114年03月薪資,
14,2025-04-20,21009161703,07410052065,16032,陳俊傑 ...,114年03月薪資,
15,2025-04-20,21009161703,22910260285,12523,蕭佳旻 ...,114年03月薪資,
16,2025-04-20,21009161703,04164037712,8843,陳育麒 ...,114年03月薪資,
17,2025-04-20,21009161703,04610633688,20620,游硯竹 ...,114年03月薪資,


📊 末三碼分布：
出帳帳戶末三碼
703    7
Name: count, dtype: int64
🔕 無資料：兆豐347 (347)，略過
🔕 無資料：兆豐335 (335)，略過
['帳務日期', '客戶/供應商', '明細', '支出', '收入', '餘額', '有無憑證(憑證編號)', '備註', '付款/尾款狀態', '應收/應付帳款發票號碼', '合約編號']
❌ 無候選交易：金額=3996（±30）
🔍 第 7 筆無法比對，略過
🔕 無資料：兆豐182 (182)，略過
🔕 無資料：兆豐697 (697)，略過


/tmp/ipython-input-2792986383.py:167: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sheet[existing] = df_sheet[existing].ffill().infer_objects(copy=False)


✅ 完成！格式化後的檔案已存為：完整金流整合.xlsx


# 憑證文件比對至更新流水帳

In [ ]:
import re
import pandas as pd
import openpyxl
from openpyxl.styles import Alignment
from google.colab import files  # 若非在 Colab 執行可移除
import zipfile # Import zipfile module
import os # Import os module

# 更新 Excel 工作表：根據模式更新憑證欄位與格式
def update_excel_sheet(ws, df, update_mode='partial'):
    df = df.astype(object).where(pd.notna(df), None).reset_index(drop=True)

    if update_mode == 'partial':
        for i, row in df.iterrows():
            value = row.get("有無憑證(憑證編號)", "")
            ws.cell(row=i + 3, column=7, value=value)

        for i, row in df.iterrows():
            cell = ws.cell(row=i + 3, column=7)
            if cell.value:
                line_count = str(cell.value).count("\n") + 1
                cell.alignment = Alignment(wrap_text=True)
                ws.row_dimensions[i + 3].height = max(15, line_count * 15)

# 驗證資料欄位是否齊全
def validate_dataframe(df, sheet_name, required_columns):
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        print(f"❌ 分頁「{sheet_name}」缺少欄位：{missing}")
        return False
    print(f"✅ 分頁「{sheet_name}」欄位完整")
    return True

# 清理檔名：移除括號尾碼（如 "(1)"）
def clean_filename(name):
    return re.sub(r"\s*\(\d+\)", "", name)

# 解析民國日期格式（如 1130205 → 2024-02-05）
def parse_minguo_date(minguo_str):
    try:
        year = int(minguo_str[:3]) + 1911
        month = int(minguo_str[3:5])
        day = int(minguo_str[5:7])
        return pd.Timestamp(year=year, month=month, day=day)
    except:
        return pd.NaT

# 通用日期解析：自動判斷民國或西元格式
def parse_row_date(val):
    try:
        val = str(val).strip()
        if len(val) == 7 and val.isdigit():  # 民國格式
            return parse_minguo_date(val)
        return pd.to_datetime(val, errors="coerce")
    except:
        return pd.NaT

# 解析憑證檔案名稱：提取憑證日期、銀行代碼、金額等資訊
def parse_voucher_filenames(filenames):
    voucher_map = {}
    for raw_name in filenames:
        name = clean_filename(raw_name)
        parts = name.split("_")
        if len(parts) < 4:
            continue

        raw_code = parts[0]
        code = raw_code.strip()

        # 民國年轉西元年
        if len(code) == 7:  # 民國年月日格式
            try:
                year = int(code[:3]) + 1911
                date_str = f"{year}{code[3:]}"
                voucher_date = pd.to_datetime(date_str, format="%Y%m%d", errors="coerce")
            except Exception:
                voucher_date = pd.NaT
        else:
            voucher_date = pd.NaT

        if pd.isna(voucher_date):
            print(f"⚠️ 檔案 {raw_name} 的日期碼「{code}」無法解析為有效日期")
            continue

        bank_code = parts[1]
        try:
            amount = int(parts[3].split(".")[0])
        except Exception:
            print(f"⚠️ 檔案 {raw_name} 的金額欄位無法解析")
            continue

        voucher_map.setdefault(bank_code, []).append((code, amount, raw_name, voucher_date))
    return voucher_map

# 建立交易索引表
def build_transaction_index(df, bank_code, tolerance=30):
    index = {}
    for i, row in df.iterrows():
        date = pd.to_datetime(row["帳務日期"], errors="coerce")
        if pd.isna(date):
            continue
        amount = row["支出"] if pd.notna(row["支出"]) else row["收入"]
        if pd.isna(amount):
            continue
        base_key = (bank_code, date.normalize(), int(amount))
        index.setdefault(base_key, []).append(i)
        # 加入誤差範圍
        for delta in range(-tolerance, tolerance + 1):
            fuzzy_key = (bank_code, date.normalize(), int(amount) + delta)
            index.setdefault(fuzzy_key, []).append(i)
    return index

# 比對憑證與交易資料：金額、日期、銀行名稱是否相符
def match_voucher(amount, row, bank_code, voucher_date, tolerance=30):
    # 內部函式：處理字串並轉成 float
    def parse_amount(val):
        try:
            return float(str(val).replace(",", "").strip())
        except:
            return None

    支出 = parse_amount(row["支出"])
    收入 = parse_amount(row["收入"])
    row_date = pd.to_datetime(row["帳務日期"], errors="coerce")
    row_bank = str(row["客戶/供應商"]).strip() if row["客戶/供應商"] else ""

    #── 檢查金額是否為 None ──
    if 支出 is None and 收入 is None:
        print(f"⚠️ 金額為空：支出={row.get('支出')}｜收入={row.get('收入')}")

    #── 檢查帳務日期是否為 NaT ──
    if pd.isna(row_date):
        print(f"⚠️ 帳務日期解析失敗：原始值={row.get('帳務日期')}")

    #── 檢查銀行代碼是否可能不一致 ──
    print(f"🏦 銀行代碼比對：憑證={bank_code}｜交易={row_bank}")

    #── 金額比對 ──
    amt_ok = (
        支出 is not None and abs(支出 - amount) <= tolerance
    ) or (
        收入 is not None and abs(收入 - amount) <= tolerance
    )
    if not amt_ok:
        print(f"❌ 金額不符：憑證金額={amount}｜支出={支出}｜收入={收入}")

    #── 日期比對 ──
    date_ok = pd.notna(row_date) and row_date.normalize() == voucher_date.normalize()
    if not date_ok:
        print(f"❌ 日期不符：憑證日期={voucher_date}｜帳務日期={row_date}")

    #── 銀行代碼比對（目前不啟用） ──
    bank_ok = bank_code in row_bank or bank_code in str(row.get("明細", "")).lower()

    return amt_ok and date_ok and bank_ok

# 主流程
def fill_voucher_reference(wb):
    global voucher_trace
    import os
    print("📁 請上傳所有憑證 PDF 或圖片（檔名需包含：憑證日期_銀行帳戶_明細_金額）")
    uploaded = files.upload()
    filenames = list(uploaded.keys())

    voucher_map = parse_voucher_filenames(filenames)
    voucher_counters = {}
    voucher_trace = {}

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]

        #── 讀取工作表內容 ──
        data = []
        for row in ws.iter_rows(min_row=3, max_row=ws.max_row, min_col=1, max_col=7):
            data.append([cell.value if cell.value not in (None, "") else None for cell in row])
        df = pd.DataFrame(data, columns=[
            "帳務日期", "客戶/供應商", "明細", "支出", "收入", "餘額", "有無憑證(憑證編號)"
        ])
        required_cols = ["帳務日期", "客戶/供應商", "明細", "支出", "收入", "餘額", "有無憑證(憑證編號)"]
        if not validate_dataframe(df, sheet_name, required_cols):
            continue

        df = df.where(pd.notna(df), None)
        df["有無憑證(憑證編號)"] = df["有無憑證(憑證編號)"].fillna("").astype(str)

        def clean_amount_column(col):
            return pd.to_numeric(
                col.astype(str)
                   .str.replace(",", "", regex=False)
                   .str.replace("元", "", regex=False)
                   .str.replace("NT$", "", regex=False)
                   .str.strip(),
                errors='coerce'
            )

        df["支出"] = clean_amount_column(df["支出"])
        df["收入"] = clean_amount_column(df["收入"])

        print(f"🔍 分頁「{sheet_name}」金額欄位型態：")
        print(df[["支出", "收入"]].dtypes)
        print(df[["支出", "收入"]].head(5))

        if sheet_name not in voucher_map:
            print(f"⚠️ 分頁「{sheet_name}」無對應憑證檔案（請確認檔名中的銀行分頁）")
            continue

        #── 建立交易索引表 ──
        transaction_index = {}
        for i, row in df.iterrows():
            date = pd.to_datetime(row["帳務日期"], errors="coerce")
            if pd.isna(date):
                continue
            amount = row["支出"] if pd.notna(row["支出"]) else row["收入"]
            if pd.isna(amount):
                continue
            for delta in range(-30, 31):  # 金額誤差 ±30
                key = (sheet_name, date.normalize(), int(amount) + delta)
                transaction_index.setdefault(key, []).append(i)

        filled_count = 0
        for code, amt, fname, voucher_date in voucher_map[sheet_name]:
            print(f"🔍 檔案：{fname} → code: '{code}' → counter_key: '{sheet_name}-{code}'")
            counter_key = f"{sheet_name}-{code}"
            serial = voucher_counters.get(counter_key, 1)
            voucher_code = f"{counter_key}{str(serial).zfill(2)}"
            voucher_counters[counter_key] = serial + 1

            key = (sheet_name, voucher_date.normalize(), int(amt))
            matched_indices = transaction_index.get(key, [])

            if not matched_indices:
                print(f"⚠️ [{sheet_name}] 金額 {amt} 無法對應憑證 {fname}")
                voucher_trace[fname] = {
                    "filled": False,
                    "code": voucher_code,
                    "sheet": sheet_name,
                    "row_index": None,
                    "original_name": fname,
                    "actual_name": None
                }
                continue

            for idx in matched_indices:
                prev = df.at[idx, "有無憑證(憑證編號)"]
                if prev and not str(prev).startswith(sheet_name):
                    prev = ""
                if voucher_code not in str(prev):
                    df.at[idx, "有無憑證(憑證編號)"] = (
                        f"{prev}\n{voucher_code}" if prev else voucher_code
                    )
                    filled_count += 1

                if fname not in voucher_trace:
                    voucher_trace[fname] = {
                        "filled": True,
                        "code": voucher_code,
                        "sheet": sheet_name,
                        "row_index": int(idx),
                        "original_name": fname,
                        "actual_name": None
                    }

        print(f"📊 分頁「{sheet_name}」欄位：{df.columns.tolist()}")
        if "餘額" not in df.columns:
            print(f"⚠️ 分頁「{sheet_name}」缺少『餘額』欄位，請檢查來源資料")
        else:
            non_empty_balance = df["餘額"].notna().sum()
            print(f"🔍 分頁「{sheet_name}」餘額欄非空筆數：{non_empty_balance}")

        print(f"📌 分頁「{sheet_name}」完成憑證填寫，共填入 {filled_count} 筆\n")

        #── 清空原有憑證欄位 ──
        for r in range(3, ws.max_row + 1):
            ws.cell(row=r, column=7, value=None)

        #── 寫回 Excel ──
        from openpyxl.styles import numbers

        currency_columns = {"支出": 4, "收入": 5, "餘額": 6}  # 對應 Excel 欄位位置

        for i, row in df.iterrows():
            for j, col in enumerate(df.columns, start=1):
                cell = ws.cell(row=i + 3, column=j)

                if col in currency_columns and row[col] not in (None, ""):
                    try:
                        raw_val = str(row[col]).replace(",", "").replace("元", "").replace("NT$", "").strip()
                        num_val = float(raw_val)
                        cell.value = num_val
                        cell.number_format = numbers.FORMAT_CURRENCY_NT  # NT$#,##0.00
                    except:
                        cell.value = row[col]  # fallback
                else:
                    cell.value = row[col] if row[col] is not None else None

        #── 自動換行與列高調整 ──
        for i, row in df.iterrows():
            cell = ws.cell(row=i + 3, column=7)
            if cell.value:
                line_count = str(cell.value).count("\n") + 1
                cell.alignment = Alignment(wrap_text=True)
                ws.row_dimensions[i + 3].height = max(15, line_count * 15)

        multi_voucher_rows = df[df["有無憑證(憑證編號)"].str.count("\n") >= 1]
        if not multi_voucher_rows.empty:
            print(f"📊 分頁「{sheet_name}」有 {len(multi_voucher_rows)} 筆交易填入了多筆憑證")

        update_excel_sheet(ws, df, update_mode='partial')

    #── 最終對照表 ──
    print("📋 檔案名稱與憑證編號對照表：")
    for fname, info in voucher_trace.items():
        status = "✅ 已填入" if info["filled"] else "⚠️ 未填入"
        print(f"{status} 📄 {fname} → 🧾 {info['code']}")

    #── 自動重新命名已填入的檔案 ──
    print("\n📦 開始重新命名已填入的憑證檔案：")

    def clean_filename(fname):
        return os.path.splitext(fname)[0]

    for fname, info in voucher_trace.items():
        if not info["filled"]:
            continue

        original_name = clean_filename(fname)
        parts = original_name.split("_")
        if len(parts) < 4:
            print(f"⚠️ 檔名格式不符，略過：{fname}")
            continue

        detail = parts[2]
        amount = parts[3]
        ext = os.path.splitext(fname)[1]
        new_name = f"{info['code']}_{detail}_{amount}{ext}"

        try:
            os.rename(fname, new_name)
            voucher_trace[fname]["actual_name"] = new_name

            print(f"✅ {fname} → {new_name}")
        except Exception as e:
            print(f"⚠️ 無法重新命名 {fname}：{e}")

    #── 金額欄位格式化（三位一撇） ──
    def clean_and_format(val):
        if pd.isna(val) or str(val).strip() == "":
            return None
        try:
            val = str(val).replace(",", "").replace("元", "").replace("NT$", "").strip()
            return f"{int(float(val)):,}"
        except:
            return val

    #── 套用格式化到金額欄位 ──
    for col in ["支出", "收入", "餘額"]:
        if col in df.columns:
            df[col] = df[col].apply(clean_and_format)

# 執行流程
wb = openpyxl.load_workbook("完整金流整合.xlsx")
fill_voucher_reference(wb)
wb.save("完整金流整合_已填憑證.xlsx")
print("✔️ 已儲存：完整金流整合_已填憑證.xlsx")

#── 壓縮成 ZIP ──
zip_name = "4月份已填憑證.zip"
with zipfile.ZipFile(zip_name, "w") as zipf:
    for _, info in voucher_trace.items():
        if "actual_name" not in info:
            print(f"⚠️ 憑證 {info.get('original_name')} 沒有重新命名紀錄，略過壓縮")
            continue  # 跳過這筆

        renamed_file = info.get("actual_name")
        if renamed_file and os.path.exists(renamed_file):
            zipf.write(renamed_file, arcname=renamed_file)
        else:
            print(f"⚠️ 檔案不存在：{renamed_file}")

print(f"\n✅ 所有檔案已搬移並打包完成：{zip_name}")

#── Colab 下載按鈕 ──
try:
    from google.colab import files
    files.download(zip_name)
except:
    print("📎 若非在 Colab 執行，請手動下載壓縮檔")

📁 請上傳所有憑證 PDF 或圖片（檔名需包含：憑證日期_銀行帳戶_明細_金額）


Saving 11404_玉山187_4月綜合對帳單.pdf to 11404_玉山187_4月綜合對帳單.pdf
Saving 1140401_計程車乘車證明(1)_195.pdf to 1140401_計程車乘車證明(1)_195.pdf
Saving 1140401_計程車乘車證明(2)_195.pdf to 1140401_計程車乘車證明(2)_195.pdf
Saving 1140403_免用統一發票收據_980.pdf to 1140403_免用統一發票收據_980.pdf
Saving 1140415_玉山187_中華電信繳費結果_999.png to 1140415_玉山187_中華電信繳費結果_999.png
Saving 1140416_玉山187_中華電信交易結果_4125.pdf to 1140416_玉山187_中華電信交易結果_4125.pdf
✅ 分頁「玉山187」欄位完整
🔍 分頁「玉山187」金額欄位型態：
支出    float64
收入    float64
dtype: object
          支出         收入
0     7421.0        0.0
1  2000000.0        0.0
2        0.0  2000000.0
3        0.0  1000000.0
4  1575000.0        0.0
🔍 檔案：1140415_玉山187_中華電信繳費結果_999.png → code: '1140415' → counter_key: '玉山187-1140415'
🔍 檔案：1140416_玉山187_中華電信交易結果_4125.pdf → code: '1140416' → counter_key: '玉山187-1140416'
📊 分頁「玉山187」欄位：['帳務日期', '客戶/供應商', '明細', '支出', '收入', '餘額', '有無憑證(憑證編號)']
🔍 分頁「玉山187」餘額欄非空筆數：19
📌 分頁「玉山187」完成憑證填寫，共填入 2 筆

✅ 分頁「台新809」欄位完整
🔍 分頁「台新809」金額欄位型態：
支出    float64
收入    float64
dtype: object
   支出  收入
0 NaN NaN

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 2月憑證文件比對至更新流水帳

In [ ]:
import re
import pandas as pd
import openpyxl
from openpyxl.styles import Alignment
from google.colab import files  # 若非在 Colab 執行可移除
import zipfile # Import zipfile module
import os # Import os module

# 更新 Excel 工作表：根據模式更新憑證欄位與格式
def update_excel_sheet(ws, df, update_mode='partial'):
    df = df.astype(object).where(pd.notna(df), None).reset_index(drop=True)

    if update_mode == 'partial':
        for i, row in df.iterrows():
            value = row.get("有無憑證(憑證編號)", "")
            ws.cell(row=i + 3, column=7, value=value)

        for i, row in df.iterrows():
            cell = ws.cell(row=i + 3, column=7)
            if cell.value:
                line_count = str(cell.value).count("\n") + 1
                cell.alignment = Alignment(wrap_text=True)
                ws.row_dimensions[i + 3].height = max(15, line_count * 15)

# 驗證資料欄位是否齊全
def validate_dataframe(df, sheet_name, required_columns):
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        print(f"❌ 分頁「{sheet_name}」缺少欄位：{missing}")
        return False
    print(f"✅ 分頁「{sheet_name}」欄位完整")
    return True

# 清理檔名：移除括號尾碼（如 "(1)"）
def clean_filename(name):
    return re.sub(r"\s*\(\d+\)", "", name)

# 解析民國日期格式（如 1130205 → 2024-02-05）
def parse_minguo_date(minguo_str):
    try:
        year = int(minguo_str[:3]) + 1911
        month = int(minguo_str[3:5])
        day = int(minguo_str[5:7])
        return pd.Timestamp(year=year, month=month, day=day)
    except:
        return pd.NaT

# 通用日期解析：自動判斷民國或西元格式
def parse_row_date(val):
    try:
        val = str(val).strip()
        if len(val) == 7 and val.isdigit():  # 民國格式
            return parse_minguo_date(val)
        return pd.to_datetime(val, errors="coerce")
    except:
        return pd.NaT

# 解析憑證檔案名稱：提取憑證日期、銀行代碼、金額等資訊
def parse_voucher_filenames(filenames):
    voucher_map = {}
    for raw_name in filenames:
        name = clean_filename(raw_name)
        parts = name.split("_")
        if len(parts) < 4:
            continue
        raw_code = parts[0]
        code = re.sub(r"[^\d]", "", raw_code)
        voucher_date = parse_row_date(code)
        bank_code = parts[1]
        try:
            amount = int(parts[3].split(".")[0])
        except:
            continue
        voucher_map.setdefault(bank_code, []).append((code, amount, raw_name, voucher_date))
    return voucher_map

# 比對憑證與交易資料：金額、日期、銀行名稱是否相符
def match_voucher(amount, row, bank_code, voucher_date, tolerance=30):
    # 內部函式：處理字串並轉成 float
    def parse_amount(val):
        try:
            return float(str(val).replace(",", "").strip())
        except:
            return None

    支出 = parse_amount(row.get("支出"))
    收入 = parse_amount(row.get("收入"))
    row_date = pd.to_datetime(row.get("帳務日期"), errors="coerce")
    row_bank = row.get("客戶/供應商")

    amt_ok = (
        支出 is not None and abs(支出 - amount) <= tolerance
    ) or (
        收入 is not None and abs(收入 - amount) <= tolerance
    )

    date_ok = pd.notna(row_date) and row_date.normalize() == voucher_date.normalize()
    bank_ok = True

    return amt_ok and date_ok and bank_ok

# 主流程：填入憑證編號至 Excel 並重新命名檔案
def fill_voucher_reference(wb):
    global voucher_trace
    import os
    print("📁 請上傳所有憑證 PDF 或圖片（檔名需包含：憑證日期_銀行帳戶_明細_金額）")
    uploaded = files.upload()
    filenames = list(uploaded.keys())

    voucher_map = parse_voucher_filenames(filenames)
    voucher_counters = {}
    voucher_trace = {}

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]

        #── 讀取＋保留原始格式 ──
        data = []
        for row in ws.iter_rows(min_row=3, max_row=ws.max_row, min_col=1, max_col=7):
            data.append([cell.value if cell.value not in (None, "") else None for cell in row])
        df = pd.DataFrame(data, columns=[
            "帳務日期", "客戶/供應商", "明細", "支出", "收入", "餘額", "有無憑證(憑證編號)"
        ])
        required_cols = ["帳務日期", "客戶/供應商", "明細", "支出", "收入", "餘額", "有無憑證(憑證編號)"]
        if not validate_dataframe(df, sheet_name, required_cols):
            continue  # 跳過這個分頁，避免後續錯誤

        #── 清理空值為 None，避免寫入 0 ──
        df = df.where(pd.notna(df), None)
        df["有無憑證(憑證編號)"] = df["有無憑證(憑證編號)"].fillna("").astype(str)

        #── 檢查是否有憑證檔案 ──
        if sheet_name not in voucher_map:
            print(f"⚠️ 分頁「{sheet_name}」無對應憑證檔案（請確認檔名中的銀行分頁）")
            continue

        filled_count = 0
        for code, amt, fname, voucher_date in voucher_map[sheet_name]:
            print(f"🔍 檔案：{fname} → code: '{code}' → counter_key: '{sheet_name}-{code}'")
            counter_key = f"{sheet_name}-{code}"
            serial = voucher_counters.get(counter_key, 1)
            voucher_code = f"{counter_key}{str(serial).zfill(2)}"
            voucher_counters[counter_key] = serial + 1

            matched = df[df.apply(lambda r: match_voucher(amt, r, sheet_name, voucher_date), axis=1)]
            if matched.empty:
                print(f"⚠️ [{sheet_name}] 金額 {amt} 無法對應憑證 {fname}")
                voucher_trace[fname] = {
                    "filled": False,
                    "code": voucher_code,
                    "sheet": sheet_name,
                    "row_index": None,
                    "original_name": fname,
                    "actual_name": None  # ✅ 保證欄位存在
                }
                continue

            for idx in matched.index:
                prev = df.at[idx, "有無憑證(憑證編號)"]
                if prev and not prev.startswith(sheet_name):
                    prev = ""
                if voucher_code not in prev:
                    df.at[idx, "有無憑證(憑證編號)"] = (
                        f"{prev}\n{voucher_code}" if prev else voucher_code
                    )
                    filled_count += 1
                voucher_trace[fname] = {
                    "filled": True,
                    "code": voucher_code,
                    "sheet": sheet_name,
                    "row_index": int(matched.index[0]),
                    "original_name": fname,
                    "actual_name": None  # ✅ 先預設為 None，稍後補上
                }
                #── 檢查欄位是否完整 ──
            print(f"📊 分頁「{sheet_name}」欄位：{df.columns.tolist()}")

            if "餘額" not in df.columns:
                print(f"⚠️ 分頁「{sheet_name}」缺少『餘額』欄位，請檢查來源資料")
            else:
                non_empty_balance = df["餘額"].notna().sum()
            print(f"🔍 分頁「{sheet_name}」餘額欄非空筆數：{non_empty_balance}")

            print(f"✅ [{sheet_name}] 憑證 {fname} 填入：{voucher_code}")

        #── 清空原有內容 ──
        for r in range(3, ws.max_row + 1):
            ws.cell(row=r, column=7, value=None)  # 只清空憑證欄

        #── 寫回 Excel（保留原始格式）──
        df = df.astype(object)
        df = df.where(pd.notna(df), None)
        df = df.reset_index(drop=True)

        for i, row in df.iterrows():
            for j, col in enumerate(df.columns, start=1):
                value = row[col]
                ws.cell(row=i + 3, column=j, value=value if value is not None else None)

        #── 自動換行＋調整列高 ──
        for i, row in df.iterrows():
            cell = ws.cell(row=i + 3, column=7)
            if cell.value:
                line_count = str(cell.value).count("\n") + 1
                cell.alignment = Alignment(wrap_text=True)
                ws.row_dimensions[i + 3].height = max(15, line_count * 15)

        print(f"📌 分頁「{sheet_name}」完成憑證填寫，共填入 {filled_count} 筆\n")
        multi_voucher_rows = df[df["有無憑證(憑證編號)"].str.count("\n") >= 1]
        if not multi_voucher_rows.empty:
            print(f"📊 分頁「{sheet_name}」有 {len(multi_voucher_rows)} 筆交易填入了多筆憑證")
        update_excel_sheet(ws, df, update_mode='partial')

    #── 最終對照表 ──
    print("📋 檔案名稱與憑證編號對照表：")
    for fname, info in voucher_trace.items():
        status = "✅ 已填入" if info["filled"] else "⚠️ 未填入"
        print(f"{status} 📄 {fname} → 🧾 {info['code']}")

    #── 自動重新命名已填入的檔案 ──
    print("\n📦 開始重新命名已填入的憑證檔案：")

    def clean_filename(fname):
        return os.path.splitext(fname)[0]

    for fname, info in voucher_trace.items():
        if not info["filled"]:
            continue

        original_name = clean_filename(fname)
        parts = original_name.split("_")
        if len(parts) < 4:
            print(f"⚠️ 檔名格式不符，略過：{fname}")
            continue

        detail = parts[2]
        amount = parts[3]
        ext = os.path.splitext(fname)[1]
        new_name = f"{info['code']}_{detail}_{amount}{ext}"

        try:
            os.rename(fname, new_name)
            voucher_trace[fname]["actual_name"] = new_name

            print(f"✅ {fname} → {new_name}")
        except Exception as e:
            print(f"⚠️ 無法重新命名 {fname}：{e}")

# 執行流程
wb = openpyxl.load_workbook("完整金流整合.xlsx")
fill_voucher_reference(wb)
wb.save("完整金流整合_已填憑證.xlsx")
print("✔️ 已儲存：完整金流整合_已填憑證.xlsx")

#── 壓縮成 ZIP ──
zip_name = "2月份已填憑證.zip"
with zipfile.ZipFile(zip_name, "w") as zipf:
    for _, info in voucher_trace.items():
        if "actual_name" not in info:
            print(f"⚠️ 憑證 {info.get('original_name')} 沒有重新命名紀錄，略過壓縮")
            continue  # 跳過這筆

        renamed_file = info.get("actual_name")
        if renamed_file and os.path.exists(renamed_file):
            zipf.write(renamed_file, arcname=renamed_file)
        else:
            print(f"⚠️ 檔案不存在：{renamed_file}")

print(f"\n✅ 所有檔案已搬移並打包完成：{zip_name}")

#── Colab 下載按鈕 ──
try:
    from google.colab import files
    files.download(zip_name)
except:
    print("📎 若非在 Colab 執行，請手動下載壓縮檔")

# 查帳助理

In [ ]:
from ipywidgets import FileUpload, Text, Button, Output, VBox, Layout, HTML
from IPython.display import display
import pandas as pd
import io
import re
from datetime import datetime

# 介面元件初始化
檔案上傳器 = FileUpload(accept='.xlsx,.csv', multiple=False)
查詢輸入框 = Text(
    placeholder='例如「兆豐 3月」、「999 玉山 3/17」',
    description='查詢語句：',
    layout=Layout(width='500px')
)
查詢按鈕 = Button(description='🔎 開始查詢', button_style='info')
回應區 = Output()

# 暫存資料
上傳資料表 = {}

# 銀行別名表
銀行別名表 = {
    '玉山': ['玉山', '玉山銀行', 'ESUN'],
    '臺銀': ['臺銀', '台銀', '臺灣銀行'],
    '國世銀': ['國世銀', '國泰', '世銀', 'CATHAY'],
    '兆豐': ['兆豐', '兆豐銀行', 'MEGA'],
    '台新': ['台新', '台新銀行'],
    '合作': ['合作', '合作金庫'],
    '中信': ['中信', '中國信託']
}

def 前處理模組(df):
    df = df.copy()

    def 清理(series):
        return (
            series.astype(str)
                  .str.replace("元", "", regex=False)
                  .str.replace(",", "", regex=False)
                  .str.extract(r"(\d+\.?\d*)")[0]
                  .astype(float)
                  .fillna(0)
                  .round()
                  .astype(int)
        )

    for col in ['支出', '收入', '餘額']:
        if col in df.columns:
            df[col] = 清理(df[col])

    if '帳務日期' in df.columns:
        df['帳務日期'] = pd.to_datetime(df['帳務日期'], errors='coerce')

    # 🔍 多欄位掃描銀行來源
    可能來源欄位 = ['客戶/供應商', '明細','備註']
    來源值 = []

    for col in 可能來源欄位:
        if col in df.columns:
            來源值.append(df[col].astype(str))

    combined = 來源值[0] if 來源值 else pd.Series([''] * len(df))
    for s in 來源值[1:]:
        combined += ' ' + s

    # 🏷️ 加入來源帳戶（工作表名稱）中的銀行別名
    if '來源帳戶' in df.columns:
        combined += ' ' + df['來源帳戶'].astype(str)

    # 統一為大寫字串
    df['銀行來源簡化'] = combined.str.upper()

    # ✅ 重設索引以確保唯一性
    df = df.reset_index(drop=True)

    return df

# 語句解析
def 解析語句_進階(語句):
    def 拆(句):
        u = 句.upper()
        if 'AND' in u:
            return [s.strip() for s in u.split('AND')], 'AND'
        if 'OR' in u:
            return [s.strip() for s in u.split('OR')], 'OR'
        return [句], '單一'

    def 單句(句):
        txt = str(句).strip().upper()
        amt, date_cond, banks, target = None, None, [], '來源'
        # 方向
        if txt.startswith('來源:'):
            target, txt = '來源', txt[3:].strip()
        elif txt.startswith('對象:'):
            target, txt = '對象', txt[3:].strip()
        # 金額
        m = re.search(r'(\d{3,6})(?:元|塊)?', txt)
        if m:
            amt = float(m.group(2))
        # 日期
        dm = re.search(r'(\d{1,2})月(\d{1,2})日', txt)
        if dm:
            y = datetime.today().year
            date_cond = pd.to_datetime(f"{y}-{dm.group(1)}-{dm.group(2)}")
        m1 = re.search(r'(\d{1,2})[/-](\d{1,2})', txt)
        if m1:
            y = datetime.today().year
            date_cond = pd.to_datetime(f"{y}-{m1.group(1)}-{m1.group(2)}")
        for m in range(1, 13):
            if re.search(fr'[{m}{["一","二","三","四","五","六","七","八","九","十","十一","十二"][m-1]}]月', txt):
                date_cond = m
                break
        # 銀行
        for bank, aliases in 銀行別名表.items():
            if any(a.upper() in txt for a in aliases):
                banks.append(bank.upper())
        return amt, date_cond, banks, target

    subs, logic = 拆(語句)
    return [單句(s) for s in subs], logic

# 交易分類
def 分類交易(df, amt, date_cond, banks, target):
    data = df.copy()

    # 金額過濾
    if amt is not None:
        data = data[(data['收入'] >= amt) | (data['支出'] >= amt)]

    # 日期過濾
    if isinstance(date_cond, pd.Timestamp):
        data = data[data['帳務日期'] == date_cond]
    elif isinstance(date_cond, int):
        if '帳務日期' in data.columns and pd.api.types.is_datetime64_any_dtype(data['帳務日期']):
            data = data[data['帳務日期'].dt.month == date_cond]
        else:
            raise ValueError("❌ 查詢失敗：帳務日期欄位不是有效的時間格式")

    # 銀行過濾
    col = '銀行來源簡化' if target == '來源' else '客戶/供應商'
    if banks:
        data = data[data[col].apply(lambda v: any(b in str(v).upper() for b in banks))]

    # 回傳：符合條件的交易 + 銀行相關交易（不一定符合其他條件）
    return data, df[df[col].apply(lambda v: any(b in str(v).upper() for b in banks))]

#  查詢邏輯
def 執行語句查詢邏輯(conds, logic, df):
    結果集 = [分類交易(df, *c) for c in conds]
    if logic == 'AND':
        dfs = [r[0] for r in 結果集 if not r[0].empty]
        if not dfs:
            return pd.DataFrame(columns=df.columns)
        res = dfs[0]
        for d in dfs[1:]:
            res = res.merge(d, how='inner')
        return res.reset_index(drop=True)  # ✅ AND 結果也重設索引
    if logic == 'OR':
        all_ = pd.concat([r[0] for r in 結果集], ignore_index=True)
        return all_.drop_duplicates().reset_index(drop=True)  # ✅ OR 結果重設索引
    return 結果集[0][0].reset_index(drop=True)  # ✅ 單一條件也重設索引

# 上傳處理
def 處理上傳變更(change):
    global 上傳資料表
    回應區.clear_output()

    for name, info in 檔案上傳器.value.items():
        buf = io.BytesIO(info['content'])

        try:
            if name.endswith('.xlsx'):
                # 1. 讀取所有分頁 → dict of DataFrame
                sheets = pd.read_excel(buf, sheet_name=None, header=None)

                # 2. 處理每個分頁：重設欄位名稱 + 加入來源帳戶
                for sheet_name, df_sheet in sheets.items():
                    df_sheet.columns = df_sheet.iloc[1]  # 第二列作為欄位名稱
                    df_sheet = df_sheet.iloc[2:].reset_index(drop=True)
                    df_sheet['來源帳戶'] = sheet_name
                    sheets[sheet_name] = df_sheet  # 更新回 dict

                # 3. 合併所有分頁
                df = pd.concat(sheets.values(), ignore_index=True, join='outer')
                df = df.reset_index(drop=True)  # ✅ 合併後重設索引，避免後續 reindex 錯誤

            else:
                # 單一檔案格式（CSV）
                df = pd.read_csv(buf)

            # 4. 儲存到全域變數
            上傳資料表['df'] = df
            print(f"✅ 上傳完成：{name}（共 {len(df)} 筆）")

        except Exception as e:
            print(f"⚠️ 上傳失敗：{e}")
            try:
                print("📊 index 是否唯一：", df.index.is_unique)
                print("📊 index 類型：", type(df.index))
                print("📊 index 範例：", df.index[:10].tolist())
            except:
                print("📌 無法檢查 index，df 尚未成功建立")

def 執行查詢(_):
    df = 上傳資料表.get('df')
    if df is None:
        with 回應區:
            回應區.clear_output()
            print("⚠️ 尚未載入資料表")
        return

    df = 前處理模組(df)
    q = 查詢輸入框.value.strip()
    if not q:
        with 回應區:
            回應區.clear_output()
            print("⚠️ 請輸入查詢語句")
        return

    try:
        conds, logic = 解析語句_進階(q)
        result = 執行語句查詢邏輯(conds, logic, df)

        result = result.drop(columns=['銀行來源簡化'], errors='ignore')

        money_cols = ['支出', '收入', '餘額']
        styled = (
            result.style
                  .format({c: '{:,.0f}' for c in money_cols})
                  .set_properties(**{'text-align': 'left'})
                  .map(lambda v: 'background-color: #648DB3' if isinstance(v, (int, float)) and v > 100000 else '', subset=['支出', '收入'])
                  .map(lambda v: 'color: #D2665A' if '兆豐' in str(v) else '', subset=['來源帳戶'])  # ✅ 改用來源帳戶來標示銀行
        )
        with 回應區:
            回應區.clear_output()
            print(f"🔎 查詢語句：{q}")
            print(f"🎯 最終結果筆數：{len(result)}")
            if not result.empty:
                # 🔍 顯示合約與明細對應欄位
                print("📎 交易對應明細如下：")
                print(result[['帳務日期', '支出', '收入', '來源帳戶', '明細']])

                # 💡 顯示格式化表格
                display(styled)
            else:
                print("⚠️ 查無符合條件的交易資料。")

    except Exception as e:
        with 回應區:
            回應區.clear_output()
            print("❌ 查詢過程發生錯誤：", str(e))
        return

# 監聽事件 & 組介面
檔案上傳器.observe(處理上傳變更, names='value')
查詢按鈕.on_click(執行查詢)
查詢輸入框.on_submit(執行查詢)

總介面 = VBox([
    HTML('<h3>📁 上傳銀行帳務檔案</h3>'),
    檔案上傳器,
    HTML('<h3>💬 輸入查詢語句</h3>'),
    查詢輸入框,
    查詢按鈕,
    回應區
])
display(總介面)